# 🔬 Benchmark: Weighting Strategies & PCA Variance Analysis

This notebook provides an econometric benchmark comparing the **Deterministic Linear Weighting Strategy** against the multivariate **PCA Statistical Weighting Strategy**.

### Key Analysis Components:
1. **Principal Component Analysis (PCA)**: Explained variance ratio and cumulative variance across 7 dimensions.
2. **Factor Loadings Heatmap**: Dimension weights derived from principal component eigenvectors.
3. **Comparative Distribution Analysis**: Deterministic vs PCA score comparisons.


In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Ensure project root is in path
sys.path.insert(0, os.path.abspath(".."))

from src.domain.access_index.dimension_calculators import FAIDimensionPipeline
from src.domain.access_index.weighting import DeterministicLinearWeightingStrategy, PCAStatisticalWeightingStrategy
from src.domain.access_index.dimensions import WeightVector
from src.domain.shared.value_objects import DimensionKey

sns.set_theme(style="whitegrid")


## 1. Load Telemetry & Compute Dimension Matrix

In [ ]:
dataset_path = os.path.join("..", "data", "synthetic", "profiles_dataset.csv")
df_raw = pd.read_csv(dataset_path)

pipeline = FAIDimensionPipeline()
dim_matrix = []

for _, row in df_raw.iterrows():
    dims = pipeline.compute_all_dimensions(row.to_dict())
    dim_matrix.append({k.value: float(v.score.value) for k, v in dims.items()})

df_dims = pd.DataFrame(dim_matrix)
print("Dimension Matrix Shape:", df_dims.shape)
df_dims.describe()


## 2. Principal Component Analysis (PCA) Variance Analysis

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_dims)

pca = PCA(n_components=7)
X_pca = pca.fit_transform(X_scaled)

explained_variance = pca.explained_variance_ratio_
cum_variance = np.cumsum(explained_variance)

df_pca_var = pd.DataFrame({
    "Component": [f"PC{i+1}" for i in range(7)],
    "Explained Variance Ratio": explained_variance,
    "Cumulative Variance": cum_variance,
})

print(df_pca_var.to_string(index=False))

fig, ax1 = plt.subplots(figsize=(9, 5))

color = 'tab:indigo'
ax1.set_xlabel('Principal Components', fontweight='bold')
ax1.set_ylabel('Explained Variance Ratio', color=color, fontweight='bold')
ax1.bar(df_pca_var['Component'], df_pca_var['Explained Variance Ratio'], color=color, alpha=0.6)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Cumulative Variance', color=color, fontweight='bold')
ax2.plot(df_pca_var['Component'], df_pca_var['Cumulative Variance'], color=color, marker='o', linewidth=2.5)
ax2.tick_params(axis='y', labelcolor=color)

plt.title("PCA Explained Variance & Cumulative Curve across 7 FAI Dimensions", fontweight='bold')
plt.show()


## 3. PCA Factor Loadings (Dimension Weight Coefficients)

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f"PC{i+1}" for i in range(7)],
    index=[k.value.capitalize() for k in DimensionKey]
)

plt.figure(figsize=(10, 6))
sns.heatmap(loadings, annot=True, fmt=".3f", cmap="coolwarm", center=0)
plt.title("PCA Eigenvector Factor Loadings Heatmap", fontweight="bold", fontsize=14)
plt.ylabel("FAI Dimension")
plt.show()


## 4. Scoring Strategy Comparison: Deterministic Linear vs PCA Weighted

In [ ]:
linear_strategy = DeterministicLinearWeightingStrategy()
pca_strategy = PCAStatisticalWeightingStrategy()
weights_equal = WeightVector.default_equal_weights()

# Derive PCA weight vector proportional to PC1 variance contribution
pc1_weights_raw = np.abs(pca.components_[0])
pc1_weights_norm = pc1_weights_raw / np.sum(pc1_weights_raw)
pca_weights_dict = {dim: Decimal(str(round(pc1_weights_norm[i], 4))) for i, dim in enumerate(DimensionKey)}
# Adjust residual to sum exactly to 1.0
residual = Decimal("1.0") - sum(pca_weights_dict.values())
pca_weights_dict[DimensionKey.ACCESS] += residual
pca_weight_vector = WeightVector(weights=pca_weights_dict)

linear_scores = []
pca_scores = []

for _, row in df_raw.iterrows():
    dims = pipeline.compute_all_dimensions(row.to_dict())
    score_lin = linear_strategy.compute_composite_score(dims, weights_equal)
    score_pca = pca_strategy.compute_composite_score(dims, pca_weight_vector)
    
    linear_scores.append(float(score_lin.value))
    pca_scores.append(float(score_pca.value))

df_comp = pd.DataFrame({
    "Deterministic Linear": linear_scores,
    "PCA Variance Weighted": pca_scores,
    "Archetype": df_raw["archetype"]
})

plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_comp, x="Deterministic Linear", fill=True, color="blue", label="Deterministic Linear", alpha=0.4)
sns.kdeplot(data=df_comp, x="PCA Variance Weighted", fill=True, color="green", label="PCA Variance Weighted", alpha=0.4)
plt.title("Distribution Comparison: Deterministic Linear vs PCA Statistical Strategy", fontweight="bold", fontsize=13)
plt.xlabel("Composite FAI Score [0.00, 100.00]")
plt.ylabel("Density")
plt.legend()
plt.show()

print("Descriptive statistics comparison:")
df_comp[["Deterministic Linear", "PCA Variance Weighted"]].describe()
